In [2]:
import os

# 1. Check if the OS environment picked it up
my_api_key = os.environ.get("LLM_API_KEY")

# 2. EMERGENCY FALLBACK: If it's empty, paste it here just to get moving 
# (Remember to delete it before pushing your code to GitHub!)
if not my_api_key:
    # os.environ["LLM_API_KEY"] = "sk-or-your-actual-key-here"
    # my_api_key = os.environ.get("LLM_API_KEY")
    pass

if my_api_key:
    print("✅ Success! Your API key was found safely in the environment.")
    print(f"Key preview: {my_api_key[:7]}...") 
else:
    print("❌ Error: LLM_API_KEY environment variable not detected.")

✅ Success! Your API key was found safely in the environment.
Key preview: sk-or-v...


In [ ]:
# ============================================================
# PART 4 - TRACK C
# MODEL PREDICTION EXPLANATION PIPELINE
# PART 2
# ============================================================

results = []

# Select three real records from the processed dataset
test_records = X.iloc[[0, 1, 2]].copy()

# ============================================================
# PIPELINE FUNCTION
# ============================================================

def run_pipeline(temp):

    print("\n" + "=" * 70)
    print(f"TEMPERATURE = {temp}")
    print("=" * 70)

    for i in range(len(test_records)):

        row = test_records.iloc[i]

        record_df = encode_record(row)

        # -------------------------------
        # Prediction
        # -------------------------------

        pred = model.predict(record_df)[0]

        prob = model.predict_proba(record_df)[0][1]

        feature_text = ""

        for c, v in row.items():

            feature_text += f"{c}: {v}\n"

        user_prompt = USER_PROMPT.format(

            features=feature_text,

            prediction=pred,

            probability=round(prob, 4)

        )

        print("\n")
        print("=" * 60)
        print(f"RECORD {i+1}")
        print("=" * 60)

        print("\nInput Features\n")

        print(record_df)

        print("\nPredicted Class :", pred)

        print("Predicted Probability :", round(prob, 4))

        # -------------------------------
        # PII CHECK
        # -------------------------------

        if has_pii(user_prompt):

            print("\nInput blocked : PII detected")

            results.append({

                "Record": i + 1,

                "Predicted Class": pred,

                "Probability": round(prob, 4),

                "Validation": "BLOCKED"

            })

            continue

        # -------------------------------
        # LLM CALL
        # -------------------------------

        response = call_llm(

            SYSTEM_PROMPT,

            user_prompt,

            temperature=temp

        )

        if response is None:

            print("No response received")

            results.append({

                "Record": i + 1,

                "Predicted Class": pred,

                "Probability": round(prob, 4),

                "Validation": "FAILED"

            })

            continue

        print("\nRaw LLM Output\n")

        print(response)

        # -------------------------------
        # JSON VALIDATION
        # -------------------------------

        try:

            parsed = json.loads(response.strip())

            validate(

                instance=parsed,

                schema=EXPLANATION_SCHEMA

            )

            print("\nValidation Status : PASS")

            print(json.dumps(parsed, indent=4))

            status = "PASS"

        except json.JSONDecodeError as e:

            print("\nJSON Decode Error")

            print(e)

            print("\nFallback Output")
            parsed = FALLBACK
            print(FALLBACK)

            status = "FAIL"

        except jsonschema.ValidationError as e:

            print("\nSchema Validation Error")

            print(e)

            print("\nFallback Output")
            parsed = FALLBACK
            print(FALLBACK)

            status = "FAIL"

        results.append({

            "Record": i + 1,

            "Predicted Class": pred,

            "Probability": round(prob, 4),

            "Validation": status

        })


# ============================================================
# TEMPERATURE = 0
# ============================================================

print("\n")
print("RUNNING TEMPERATURE = 0")

results.clear()

run_pipeline(0.0)

summary_temp0 = pd.DataFrame(results)

print("\nSummary Table (Temperature = 0)")

print(summary_temp0.to_string(index=False))


# ============================================================
# TEMPERATURE = 0.7
# ============================================================

print("\n")
print("RUNNING TEMPERATURE = 0.7")

results.clear()

run_pipeline(0.7)

summary_temp07 = pd.DataFrame(results)

print("\nSummary Table (Temperature = 0.7)")

print(summary_temp07.to_string(index=False))


print("\nCompleted Successfully.")

=== VERIFYING LLM BASELINE CONNECTION ===
LLM Handshake Response: "hello"

=== TESTING PII GUARDRAIL SYSTEM ===
Testing Dirty Input (Should block -> True):  True
Testing Clean Input (Should proceed -> False): False

=== RUNNING DETERMINISTIC EXPLANATIONS (TEMP = 0.0) ===
--- PROCESSING RECORD 1 (Temperature=0.0) ---
Validation outcome: PASS
Validated JSON Output:
{
  "prediction_label": "Employee is predicted to leave the company.",
  "confidence_level": "high",
  "top_reason": "The employee's very low tenure at the company (0 years) combined with a short overall career (1 total working year) indicates a high flight risk, suggesting they may be seeking more stable or advanced opportunities elsewhere.",
  "second_reason": "Frequent business travel can contribute to burnout and dissatisfaction, especially for new employees, further increasing the likelihood of attrition.",
  "next_step": "Implement a robust onboarding and mentorship program to integrate new employees, focusing on career 